In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Jul 26 02:18:55 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671188,35.9,1479551,79.1,1479551,79.1
Vcells,1242557,9.5,8388608,64.0,1978697,15.1


In [3]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 171719

PARAM$experimento <- 1133
PARAM$dataset <- "analistajr_competencia_2026.csv.gz"

PARAM$FT$semillerio <- 30  

if( !require("primes")) install.packages("primes")
require("primes")

Loading required package: primes



In [5]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

In [6]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

In [7]:
if( !require("mice")) install.packages("mice", repos = "http://cran.us.r-project.org")
require("mice")

Loading required package: mice


Attaching package: ‘mice’


The following object is masked from ‘package:stats’:

    filter


The following objects are masked from ‘package:base’:

    cbind, rbind




In [8]:
# Escrito por alumnos de  Universidad Austral  Rosario

Corregir_MICE <- function(pcampo, pmeses) {

  meth <- rep("", ncol(dataset))
  names(meth) <- colnames(dataset)
  meth[names(meth) == pcampo] <- "sample"

  # llamada a mice  !
  imputacion <- mice(dataset,
    method = meth,
    maxit = 5,
    m = 1,
    seed = 7)

  tbl <- mice::complete(dataset)

  dataset[, paste0(pcampo) := ifelse(foto_mes %in% pmeses, tbl[, get(pcampo)], get(pcampo))]

}


In [9]:
Corregir_interpolar <- function(pcampo, pmeses) {

  tbl <- dataset[, list(
    "v1" = shift(get(pcampo), 1, type = "lag"),
    "v2" = shift(get(pcampo), 1, type = "lead")
  ),
  by = eval(envg$PARAM$dataset_metadata$entity_id)
  ]

  tbl[, paste0(envg$PARAM$dataset_metadata$entity_id) := NULL]
  tbl[, promedio := rowMeans(tbl, na.rm = TRUE)]

  dataset[
    ,
    paste0(pcampo) := ifelse(!(foto_mes %in% pmeses),
      get(pcampo),
      tbl$promedio
    )
  ]
}

In [10]:
AsignarNA_campomeses <- function(pcampo, pmeses) {

  if( pcampo %in% colnames( dataset ) ) {

    dataset[ foto_mes %in% pmeses, paste0(pcampo) := NA ]
  }
}

In [11]:

Corregir_atributo <- function(pcampo, pmeses, pmetodo)
{
  # si el campo no existe en el dataset, Afuera !
  if( !(pcampo %in% colnames( dataset )) )
    return( 1 )

  # llamo a la funcion especializada que corresponde
  switch( pmetodo,
    "MachineLearning"     = AsignarNA_campomeses(pcampo, pmeses),
    "EstadisticaClasica"  = Corregir_interpolar(pcampo, pmeses),
    "MICE"                = Corregir_MICE(pcampo, pmeses),
  )

  return( 0 )
}

In [12]:

Corregir_Rotas <- function(dataset, pmetodo) {
  gc(verbose= FALSE)
  cat( "inicio Corregir_Rotas()\n")
  # acomodo los errores del dataset

  Corregir_atributo("active_quarter", c(202006), pmetodo) # 1
  Corregir_atributo("internet", c(202006), pmetodo) # 2

  Corregir_atributo("mrentabilidad", c(201905, 201910, 202006), pmetodo) # 3
  Corregir_atributo("mrentabilidad_annual", c(201905, 201910, 202006), pmetodo) # 4

  Corregir_atributo("mcomisiones", c(201905, 201910, 202006), pmetodo) # 5

  Corregir_atributo("mactivos_margen", c(201905, 201910, 202006), pmetodo) # 6
  Corregir_atributo("mpasivos_margen", c(201905, 201910, 202006), pmetodo) # 7

  Corregir_atributo("mcuentas_saldo", c(202006), pmetodo) # 8

  Corregir_atributo("ctarjeta_debito_transacciones", c(202006), pmetodo) # 9

  Corregir_atributo("mautoservicio", c(202006), pmetodo) # 10

  Corregir_atributo("ctarjeta_visa_transacciones", c(202006), pmetodo) # 11
  Corregir_atributo("mtarjeta_visa_consumo", c(202006), pmetodo) # 12

  Corregir_atributo("ctarjeta_master_transacciones", c(202006), pmetodo) # 13
  Corregir_atributo("mtarjeta_master_consumo", c(202006), pmetodo) # 14

  Corregir_atributo("ctarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 15
  Corregir_atributo("mttarjeta_visa_debitos_automaticos", c(201904), pmetodo) # 16

  Corregir_atributo("ccajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 17

  Corregir_atributo("mcajeros_propios_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 18

  Corregir_atributo("ctarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 19

  Corregir_atributo("mtarjeta_visa_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 20

  Corregir_atributo("ctarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 21

  Corregir_atributo("mtarjeta_master_descuentos",
    c(201910, 202002, 202006, 202009, 202010, 202102), pmetodo) # 22

  Corregir_atributo("ccomisiones_otras", c(201905, 201910, 202006), pmetodo) # 23
  Corregir_atributo("mcomisiones_otras", c(201905, 201910, 202006), pmetodo) # 24

  Corregir_atributo("cextraccion_autoservicio", c(202006), pmetodo) # 25
  Corregir_atributo("mextraccion_autoservicio", c(202006), pmetodo) # 26

  Corregir_atributo("ccheques_depositados", c(202006), pmetodo) # 27
  Corregir_atributo("mcheques_depositados", c(202006), pmetodo) # 28
  Corregir_atributo("ccheques_emitidos", c(202006), pmetodo) # 29
  Corregir_atributo("mcheques_emitidos", c(202006), pmetodo) # 30
  Corregir_atributo("ccheques_depositados_rechazados", c(202006), pmetodo) # 31
  Corregir_atributo("mcheques_depositados_rechazados", c(202006), pmetodo) # 32
  Corregir_atributo("ccheques_emitidos_rechazados", c(202006), pmetodo) # 33
  Corregir_atributo("mcheques_emitidos_rechazados", c(202006), pmetodo) # 34

  Corregir_atributo("tcallcenter", c(202006), pmetodo) # 35
  Corregir_atributo("ccallcenter_transacciones", c(202006), pmetodo) # 36

  Corregir_atributo("thomebanking", c(202006), pmetodo) # 37
  Corregir_atributo("chomebanking_transacciones", c(201910, 202006), pmetodo) # 38

  Corregir_atributo("ccajas_transacciones", c(202006), pmetodo) # 39
  Corregir_atributo("ccajas_consultas", c(202006), pmetodo) # 40

  Corregir_atributo("ccajas_depositos", c(202006, 202105), pmetodo) # 41

  Corregir_atributo("ccajas_extracciones", c(202006), pmetodo) # 41
  Corregir_atributo("ccajas_otras", c(202006), pmetodo) # 43

  Corregir_atributo("catm_trx", c(202006), pmetodo) # 44
  Corregir_atributo("matm", c(202006), pmetodo) # 45
  Corregir_atributo("catm_trx_other", c(202006), pmetodo) # 46
  Corregir_atributo("matm_other", c(202006), pmetodo) # 47

  cat( "fin Corregir_rotas()\n")
}


In [13]:
# resuelvo el Catastrophe Analysis

setorder( dataset, numero_de_cliente, foto_mes )

PARAM$CA$metodo= "MachineLearning"

if( PARAM$CA$metodo %in% c("MachineLearning", "EstadisticaClasica", "MICE") )
  Corregir_Rotas(dataset, PARAM$CA$metodo)

inicio Corregir_Rotas()
fin Corregir_rotas()


In [14]:
# meses que me interesan para el ajuste de variables monetarias
vfoto_mes <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107, 202108, 202109
)


In [15]:
# los valores que siguen fueron calculados por alumnos

# momento 1.0  31-dic-2020 a las 23:59
vIPC <- c(
  1.9903030878, 1.9174403544, 1.8296186587,
  1.7728862972, 1.7212488323, 1.6776304408,
  1.6431248196, 1.5814483345, 1.4947526791,
  1.4484037589, 1.3913580777, 1.3404220402,
  1.3154288912, 1.2921698342, 1.2472681797,
  1.2300475145, 1.2118694724, 1.1881073259,
  1.1693969743, 1.1375456949, 1.1065619600,
  1.0681100000, 1.0370000000, 1.0000000000,
  0.9680542110, 0.9344152616, 0.8882274350,
  0.8532444140, 0.8251880213, 0.8003763543,
  0.7763107219, 0.7566381305, 0.7289384687
)

vdolar_blue <- c(
   39.045455,  38.402500,  41.639474,
   44.274737,  46.095455,  45.063333,
   43.983333,  54.842857,  61.059524,
   65.545455,  66.750000,  72.368421,
   77.477273,  78.191667,  82.434211,
  101.087500, 126.236842, 125.857143,
  130.782609, 133.400000, 137.954545,
  170.619048, 160.400000, 153.052632,
  157.900000, 149.380952, 143.615385,
  146.250000, 153.550000, 162.000000,
  178.478261, 180.878788, 184.357143
)

vdolar_oficial <- c(
   38.430000,  39.428000,  42.542105,
   44.354211,  46.088636,  44.955000,
   43.751429,  54.650476,  58.790000,
   61.403182,  63.012632,  63.011579,
   62.983636,  63.580556,  65.200000,
   67.872000,  70.047895,  72.520952,
   75.324286,  77.488500,  79.430909,
   83.134762,  85.484737,  88.181667,
   91.474000,  93.997778,  96.635909,
   98.526000,  99.613158, 100.619048,
  101.619048, 102.569048, 103.781818
)

vUVA <- c(
  2.001408838932958,  1.950325472789153,  1.89323032351521,
  1.8247220405493787, 1.746027787673673,  1.6871348409529485,
  1.6361678865622313, 1.5927529755859773, 1.5549162794128493,
  1.4949100586391746, 1.4197729500774545, 1.3678188186372326,
  1.3136508617223726, 1.2690535173062818, 1.2381595983200178,
  1.211656735577568,  1.1770808941405335, 1.1570338657445522,
  1.1388769475653255, 1.1156993751209352, 1.093638313080772,
  1.0657171590878205, 1.0362173587708712, 1.0,
  0.9669867858358365, 0.9323750098728378, 0.8958202912590305,
  0.8631993702994263, 0.8253893405524657, 0.7928918905364516,
  0.7666323845128089, 0.7428976357662823, 0.721615762047849
)


In [16]:
tb_indices <- as.data.table( list(
  "IPC" = vIPC,
  "dolar_blue" = vdolar_blue,
  "dolar_oficial" = vdolar_oficial,
  "UVA" = vUVA
  )
)

tb_indices[[ 'foto_mes' ]] <- vfoto_mes

tb_indices

IPC,dolar_blue,dolar_oficial,UVA,foto_mes
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1.9903031,39.04545,38.43000,2.0014088,201901
1.9174404,38.40250,39.42800,1.9503255,201902
1.8296187,41.63947,42.54210,1.8932303,201903
1.7728863,44.27474,44.35421,1.8247220,201904
1.7212488,46.09546,46.08864,1.7460278,201905
1.6776304,45.06333,44.95500,1.6871348,201906
1.6431248,43.98333,43.75143,1.6361679,201907
1.5814483,54.84286,54.65048,1.5927530,201908
1.4947527,61.05952,58.79000,1.5549163,201909


In [17]:
drift_UVA <- function(campos_monetarios) {
  cat( "inicio drift_UVA()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.UVA,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_UVA()\n")
}


In [18]:
drift_dolar_oficial <- function(campos_monetarios) {
  cat( "inicio drift_dolar_oficial()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_oficial,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_oficial()\n")
}


In [19]:
drift_dolar_blue <- function(campos_monetarios) {
  cat( "inicio drift_dolar_blue()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD / i.dolar_blue,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_dolar_blue()\n")
}


In [20]:
drift_deflacion <- function(campos_monetarios) {
  cat( "inicio drift_deflacion()\n")

  dataset[tb_indices,
    on = c("foto_mes"),
    (campos_monetarios) := .SD * i.IPC,
    .SDcols = campos_monetarios
  ]

  cat( "fin drift_deflacion()\n")
}


In [21]:
drift_rank_simple <- function(campos_drift) {

  cat( "inicio drift_rank_simple()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_rank") :=
      (frank(get(campo), ties.method = "random") - 1) / (.N - 1), by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat( "fin drift_rank_simple()\n")
}


In [22]:
# El cero se transforma en cero
# los positivos se rankean por su lado
# los negativos se rankean por su lado

drift_rank_cero_fijo <- function(campos_drift) {

  cat( "inicio drift_rank_cero_fijo()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[get(campo) == 0, paste0(campo, "_rank") := 0]
    dataset[get(campo) > 0, paste0(campo, "_rank") :=
      frank(get(campo), ties.method = "random") / .N, by = list(foto_mes)]

    dataset[get(campo) < 0, paste0(campo, "_rank") :=
      -frank(-get(campo), ties.method = "random") / .N, by = list(foto_mes)]
    dataset[, (campo) := NULL]
  }
  cat("\n")
  cat( "fin drift_rank_cero_fijo()\n")
}


In [23]:
drift_estandarizar <- function(campos_drift) {

  cat( "inicio drift_estandarizar()\n")
  for (campo in campos_drift)
  {
    cat(campo, " ")
    dataset[, paste0(campo, "_normal") :=
      (get(campo) -mean(campo, na.rm=TRUE)) / sd(get(campo), na.rm=TRUE),
      by = list(foto_mes)]

    dataset[, (campo) := NULL]
  }
  cat( "fin drift_estandarizar()\n")
}


In [24]:
# por como armé los nombres de campos,
#  estos son los campos que expresan variables monetarias
campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like%
  "^(m|Visa_m|Master_m|vm_m)"]

campos_monetarios

[1] "mrentabilidad"                      "mrentabilidad_annual"              
 [3] "mcomisiones"                        "mactivos_margen"                   
 [5] "mpasivos_margen"                    "mcuenta_corriente"                 
 [7] "mcaja_ahorro"                       "mcuentas_saldo"                    
 [9] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
[11] "mprestamos_personales"              "mpayroll"                          
[13] "mttarjeta_visa_debitos_automaticos" "mcomisiones_mantenimiento"         
[15] "mtransferencias_recibidas"          "Master_mfinanciacion_limite"       
[17] "Master_msaldototal"                 "Master_mlimitecompra"              
[19] "Master_mconsumototal"               "Master_mpagominimo"                
[21] "Visa_mfinanciacion_limite"          "Visa_msaldototal"                  
[23] "Visa_mlimitecompra"                 "Visa_mconsumototal"                
[25] "Visa_mpagominimo"

In [25]:
# ejecuto el Data Drifting
setorder( dataset, numero_de_cliente, foto_mes )


PARAM$DR$metodo <- "rank_cero_fijo"

switch(PARAM$DR$metodo,
  "ninguno"        = cat("No hay correccion del data drifting"),
  "rank_simple"    = drift_rank_simple(campos_monetarios),
  "rank_cero_fijo" = drift_rank_cero_fijo(campos_monetarios),
  "deflacion"      = drift_deflacion(campos_monetarios),
  "dolar_blue"     = drift_dolarblue(campos_monetarios),
  "dolar_oficial"  = drift_dolaroficial(campos_monetarios),
  "UVA"            = drift_UVA(campos_monetarios),
  "estandarizar"   = drift_estandarizar(campos_monetarios)
)


inicio drift_rank_cero_fijo()
mrentabilidad  mrentabilidad_annual  mcomisiones  mactivos_margen  mpasivos_margen  mcuenta_corriente  mcaja_ahorro  mcuentas_saldo  mtarjeta_visa_consumo  mtarjeta_master_consumo  mprestamos_personales  mpayroll  mttarjeta_visa_debitos_automaticos  mcomisiones_mantenimiento  mtransferencias_recibidas  Master_mfinanciacion_limite  Master_msaldototal  Master_mlimitecompra  Master_mconsumototal  Master_mpagominimo  Visa_mfinanciacion_limite  Visa_msaldototal  Visa_mlimitecompra  Visa_mconsumototal  Visa_mpagominimo  
fin drift_rank_cero_fijo()


In [26]:
colnames(dataset)

[1] "numero_de_cliente"                      
 [2] "foto_mes"                               
 [3] "internet"                               
 [4] "cliente_edad"                           
 [5] "cliente_antiguedad"                     
 [6] "cproductos"                             
 [7] "cdescubierto_preacordado"               
 [8] "ctarjeta_visa"                          
 [9] "ctarjeta_visa_transacciones"            
[10] "ctarjeta_master"                        
[11] "ctarjeta_master_transacciones"          
[12] "cprestamos_personales"                  
[13] "cpayroll_trx"                           
[14] "ccomisiones_mantenimiento"              
[15] "ccomisiones_otras"                      
[16] "ccallcenter_transacciones"              
[17] "thomebanking"                           
[18] "chomebanking_transacciones"             
[19] "ctrx_quarter"                           
[20] "Master_status"                          
[21] "Master_Fvencimiento"                    
[22] "Master_fultimo_cierre"                  
[23] "Master_fechaalta"                       
[24] "Master_cconsumos"                       
[25] "Visa_status"                            
[26] "Visa_Fvencimiento"                      
[27] "Visa_fultimo_cierre"                    
[28] "Visa_fechaalta"                         
[29] "Visa_cconsumos"                         
[30] "clase_ternaria"                         
[31] "mrentabilidad_rank"                     
[32] "mrentabilidad_annual_rank"              
[33] "mcomisiones_rank"                       
[34] "mactivos_margen_rank"                   
[35] "mpasivos_margen_rank"                   
[36] "mcuenta_corriente_rank"                 
[37] "mcaja_ahorro_rank"                      
[38] "mcuentas_saldo_rank"                    
[39] "mtarjeta_visa_consumo_rank"             
[40] "mtarjeta_master_consumo_rank"           
[41] "mprestamos_personales_rank"             
[42] "mpayroll_rank"                          
[43] "mttarjeta_visa_debitos_automaticos_rank"
[44] "mcomisiones_mantenimiento_rank"         
[45] "mtransferencias_recibidas_rank"         
[46] "Master_mfinanciacion_limite_rank"       
[47] "Master_msaldototal_rank"                
[48] "Master_mlimitecompra_rank"              
[49] "Master_mconsumototal_rank"              
[50] "Master_mpagominimo_rank"                
[51] "Visa_mfinanciacion_limite_rank"         
[52] "Visa_msaldototal_rank"                  
[53] "Visa_mlimitecompra_rank"                
[54] "Visa_mconsumototal_rank"                
[55] "Visa_mpagominimo_rank"

In [27]:
library(lubridate)


Attaching package: ‘lubridate’


The following objects are masked from ‘package:data.table’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [28]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

In [29]:
AgregarVariables_IntraMes <- function(dataset) {
    cat( "inicio AgregarVariables_IntraMes()\n")
    gc(verbose= FALSE)

    # el mes 1,2, ..12
    if( atributos_presentes( c("foto_mes") ))
      dataset[, kmes := foto_mes %% 100]
    
    # variable extraida de una tesis de maestria de Irlanda
    if( atributos_presentes( c("mpayroll", "cliente_edad") ))
      dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

    # ctrx_quarter normalizado
    if( atributos_presentes( c("ctrx_quarter") ))
        dataset[, ctrx_quarter_normalizado := as.numeric(ctrx_quarter) ]

    if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
        dataset[cliente_antiguedad == 1, ctrx_quarter_normalizado := ctrx_quarter * 5]

    if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
        dataset[cliente_antiguedad == 2, ctrx_quarter_normalizado := ctrx_quarter * 2]
    
    if( atributos_presentes( c("ctrx_quarter", "cliente_antiguedad") ))
        dataset[
          cliente_antiguedad == 3,
          ctrx_quarter_normalizado := ctrx_quarter * 1.2
        ]

    #fecha normalizada
    if(atributos_presentes(c("foto_mes")))
        dataset[,foto_mes_formato_fecha := as.Date(paste(substr(dataset$foto_mes,1,4),substr(dataset$foto_mes,5,6),"01",sep='-'))]

    cols_transacciones <- grep("_transacciones$", names(dataset), value = TRUE)
    dataset[, cantidad_total_transacciones := rowSums(.SD, na.rm = TRUE), .SDcols = cols_transacciones]
    
    if(atributos_presentes(c("cantidad_total_transacciones"))) {
        auxiliarmenos1 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha, cantidad_total_transacciones)]
        auxiliarmenos2 <- dataset[,list(numero_de_cliente,foto_mes_formato_fecha,cantidad_total_transacciones)]
        
        auxiliarmenos1$foto_mes_formato_fecha <- auxiliarmenos1$foto_mes_formato_fecha  %m+%  months(1)
        auxiliarmenos2$foto_mes_formato_fecha <- auxiliarmenos2$foto_mes_formato_fecha %m+% months(2)
        
        auxiliarmenos1$codigo <- paste(auxiliarmenos1$numero_de_cliente,auxiliarmenos1$foto_mes_formato_fecha,sep='-')
        auxiliarmenos2$codigo <- paste(auxiliarmenos2$numero_de_cliente,auxiliarmenos2$foto_mes_formato_fecha,sep='-')
        
        dataset[, codigo := paste(numero_de_cliente, foto_mes_formato_fecha, sep='-') ]
        
        dataset[ auxiliarmenos1,
                on = "codigo",
                transaccionesmenos1 := i.cantidad_total_transacciones ]
        
        dataset[ auxiliarmenos2,
                on = "codigo",
                transaccionesmenos2 := i.cantidad_total_transacciones ]
        
        dataset[, cantidad_total_transacciones_quarter := rowSums(
            cbind(
                cantidad_total_transacciones,
                transaccionesmenos1,
                transaccionesmenos2
            ),
            na.rm=T
        )]
        
        dataset[, codigo := NULL ]
        dataset[, transaccionesmenos1 := NULL ]
        dataset[, transaccionesmenos2 := NULL ]
        dataset[, foto_mes_formato_fecha := NULL ]
        rm(auxiliarmenos1)
        rm(auxiliarmenos2)
    }
    
    if( atributos_presentes( c("cantidad_total_transacciones_quarter") ))
        dataset[, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter]
    
    if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
        dataset[cliente_antiguedad == 1, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 5]
    
    if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
        dataset[cliente_antiguedad == 2, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 2]
    
    if( atributos_presentes( c("cantidad_total_transacciones_quarter", "cliente_antiguedad") ))
        dataset[cliente_antiguedad == 3, cantidad_total_transacciones_quarter_normalizado := cantidad_total_transacciones_quarter * 1.2]
    
    
    # se crean los nuevos campos para MasterCard  y Visa,
    #  teniendo en cuenta los NA's
    # varias formas de combinar Visa_status y Master_status
    if( atributos_presentes( c("Master_status", "Visa_status") ))
    {
        dataset[, vm_status01 := pmax(Master_status, Visa_status, na.rm = TRUE)]
        dataset[, vm_status02 := Master_status + Visa_status]
        
        dataset[, vm_status03 := pmax(
          ifelse(is.na(Master_status), 10, Master_status),
          ifelse(is.na(Visa_status), 10, Visa_status)
        )]
        
        dataset[, vm_status04 := ifelse(is.na(Master_status), 10, Master_status)
          + ifelse(is.na(Visa_status), 10, Visa_status)]
        
        dataset[, vm_status05 := ifelse(is.na(Master_status), 10, Master_status)
          + 100 * ifelse(is.na(Visa_status), 10, Visa_status)]
        
        dataset[, vm_status06 := ifelse(is.na(Visa_status),
          ifelse(is.na(Master_status), 10, Master_status),
          Visa_status
        )]
        
        dataset[, mv_status07 := ifelse(is.na(Master_status),
          ifelse(is.na(Visa_status), 10, Visa_status),
          Master_status
        )]
    }
    
    # combino MasterCard y Visa
    if( atributos_presentes( c("Master_mfinanciacion_limite", "Visa_mfinanciacion_limite") ))
    dataset[, vm_mfinanciacion_limite := rowSums(cbind(Master_mfinanciacion_limite, Visa_mfinanciacion_limite), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_Fvencimiento", "Visa_Fvencimiento") ))
    dataset[, vm_Fvencimiento := pmin(Master_Fvencimiento, Visa_Fvencimiento, na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_Finiciomora", "Visa_Finiciomora") ))
    dataset[, vm_Finiciomora := pmin(Master_Finiciomora, Visa_Finiciomora, na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_msaldototal", "Visa_msaldototal") ))
    dataset[, vm_msaldototal := rowSums(cbind(Master_msaldototal, Visa_msaldototal), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_msaldopesos", "Visa_msaldopesos") ))
    dataset[, vm_msaldopesos := rowSums(cbind(Master_msaldopesos, Visa_msaldopesos), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_msaldodolares", "Visa_msaldodolares") ))
    dataset[, vm_msaldodolares := rowSums(cbind(Master_msaldodolares, Visa_msaldodolares), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mconsumospesos", "Visa_mconsumospesos") ))
    dataset[, vm_mconsumospesos := rowSums(cbind(Master_mconsumospesos, Visa_mconsumospesos), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mconsumosdolares", "Visa_mconsumosdolares") ))
    dataset[, vm_mconsumosdolares := rowSums(cbind(Master_mconsumosdolares, Visa_mconsumosdolares), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mlimitecompra", "Visa_mlimitecompra") ))
    dataset[, vm_mlimitecompra := rowSums(cbind(Master_mlimitecompra, Visa_mlimitecompra), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_madelantopesos", "Visa_madelantopesos") ))
    dataset[, vm_madelantopesos := rowSums(cbind(Master_madelantopesos, Visa_madelantopesos), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_madelantodolares", "Visa_madelantodolares") ))
    dataset[, vm_madelantodolares := rowSums(cbind(Master_madelantodolares, Visa_madelantodolares), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_fultimo_cierre", "Visa_fultimo_cierre") ))
    dataset[, vm_fultimo_cierre := pmax(Master_fultimo_cierre, Visa_fultimo_cierre, na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mpagado", "Visa_mpagado") ))
    dataset[, vm_mpagado := rowSums(cbind(Master_mpagado, Visa_mpagado), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mpagospesos", "Visa_mpagospesos") ))
    dataset[, vm_mpagospesos := rowSums(cbind(Master_mpagospesos, Visa_mpagospesos), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mpagosdolares", "Visa_mpagosdolares") ))
    dataset[, vm_mpagosdolares := rowSums(cbind(Master_mpagosdolares, Visa_mpagosdolares), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_fechaalta", "Visa_fechaalta") ))
    dataset[, vm_fechaalta := pmax(Master_fechaalta, Visa_fechaalta, na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mconsumototal", "Visa_mconsumototal") ))
    dataset[, vm_mconsumototal := rowSums(cbind(Master_mconsumototal, Visa_mconsumototal), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_cconsumos", "Visa_cconsumos") ))
    dataset[, vm_cconsumos := rowSums(cbind(Master_cconsumos, Visa_cconsumos), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_cadelantosefectivo", "Visa_cadelantosefectivo") ))
    dataset[, vm_cadelantosefectivo := rowSums(cbind(Master_cadelantosefectivo, Visa_cadelantosefectivo), na.rm = TRUE)]
    
    if( atributos_presentes( c("Master_mpagominimo", "Visa_mpagominimo") ))
        dataset[, vm_mpagominimo := rowSums(cbind(Master_mpagominimo, Visa_mpagominimo), na.rm = TRUE)]
    
    # a partir de aqui juego con la suma de Mastercard y Visa
    if( atributos_presentes( c("Master_mlimitecompra", "vm_mlimitecompra") ))
        dataset[, vmr_Master_mlimitecompra := Master_mlimitecompra / vm_mlimitecompra]
    
    if( atributos_presentes( c("Visa_mlimitecompra", "vm_mlimitecompra") ))
        dataset[, vmr_Visa_mlimitecompra := Visa_mlimitecompra / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_msaldototal", "vm_mlimitecompra") ))
        dataset[, vmr_msaldototal := vm_msaldototal / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_msaldopesos", "vm_mlimitecompra") ))
        dataset[, vmr_msaldopesos := vm_msaldopesos / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_msaldopesos", "vm_msaldototal") ))
        dataset[, vmr_msaldopesos2 := vm_msaldopesos / vm_msaldototal]
    
    if( atributos_presentes( c("vm_msaldodolares", "vm_mlimitecompra") ))
        dataset[, vmr_msaldodolares := vm_msaldodolares / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_msaldodolares", "vm_msaldototal") ))
        dataset[, vmr_msaldodolares2 := vm_msaldodolares / vm_msaldototal]
    
    if( atributos_presentes( c("vm_mconsumospesos", "vm_mlimitecompra") ))
        dataset[, vmr_mconsumospesos := vm_mconsumospesos / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mconsumosdolares", "vm_mlimitecompra") ))
        dataset[, vmr_mconsumosdolares := vm_mconsumosdolares / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_madelantopesos", "vm_mlimitecompra") ))
        dataset[, vmr_madelantopesos := vm_madelantopesos / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_madelantodolares", "vm_mlimitecompra") ))
        dataset[, vmr_madelantodolares := vm_madelantodolares / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mpagado", "vm_mlimitecompra") ))
        dataset[, vmr_mpagado := vm_mpagado / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mpagospesos", "vm_mlimitecompra") ))
        dataset[, vmr_mpagospesos := vm_mpagospesos / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mpagosdolares", "vm_mlimitecompra") ))
        dataset[, vmr_mpagosdolares := vm_mpagosdolares / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mconsumototal", "vm_mlimitecompra") ))
        dataset[, vmr_mconsumototal := vm_mconsumototal / vm_mlimitecompra]
    
    if( atributos_presentes( c("vm_mpagominimo", "vm_mlimitecompra") ))
        dataset[, vmr_mpagominimo := vm_mpagominimo / vm_mlimitecompra]
    
    # valvula de seguridad para evitar valores infinitos
    # paso los infinitos a NULOS
    infinitos <- lapply(
    names(dataset),
    function(.name) dataset[, sum(is.infinite(get(.name)))]
    )
    
    infinitos_qty <- sum(unlist(infinitos))
    if (infinitos_qty > 0) {
        cat(
          "ATENCION, hay", infinitos_qty,
          "valores infinitos en tu dataset. Seran pasados a NA\n"
        )
        for (j in names(dataset)) set(dataset, which(is.infinite(dataset[[j]])), j, NA)
    }
    
    
    # valvula de seguridad para evitar valores NaN  que es 0/0
    # paso los NaN a 0 , decision polemica si las hay
    # se invita a asignar un valor razonable segun la semantica del campo creado
    nans <- lapply(
        names(dataset),
        function(.name) dataset[, sum(is.nan(get(.name)))]
    )
    
    nans_qty <- sum(unlist(nans))
    if (nans_qty > 0) {
        cat(
          "ATENCION, hay", nans_qty,
          "valores NaN 0/0 en tu dataset. Seran pasados arbitrariamente a 0\n"
        )
    
        cat("Si no te gusta la decision, modifica a gusto el programa!\n\n")
        for (j in names(dataset)) set(dataset, which(is.nan(dataset[[j]])), j, 0)
    }
    
    cat( "fin AgregarVariables_IntraMes()\n")
}

In [30]:
AgregarVariables_IntraMes(dataset)

inicio AgregarVariables_IntraMes()
fin AgregarVariables_IntraMes()


In [31]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                               
 [2] "foto_mes"                                        
 [3] "internet"                                        
 [4] "cliente_edad"                                    
 [5] "cliente_antiguedad"                              
 [6] "cproductos"                                      
 [7] "cdescubierto_preacordado"                        
 [8] "ctarjeta_visa"                                   
 [9] "ctarjeta_visa_transacciones"                     
[10] "ctarjeta_master"                                 
[11] "ctarjeta_master_transacciones"                   
[12] "cprestamos_personales"                           
[13] "cpayroll_trx"                                    
[14] "ccomisiones_mantenimiento"                       
[15] "ccomisiones_otras"                               
[16] "ccallcenter_transacciones"                       
[17] "thomebanking"                                    
[18] "chomebanking_transacciones"                      
[19] "ctrx_quarter"                                    
[20] "Master_status"                                   
[21] "Master_Fvencimiento"                             
[22] "Master_fultimo_cierre"                           
[23] "Master_fechaalta"                                
[24] "Master_cconsumos"                                
[25] "Visa_status"                                     
[26] "Visa_Fvencimiento"                               
[27] "Visa_fultimo_cierre"                             
[28] "Visa_fechaalta"                                  
[29] "Visa_cconsumos"                                  
[30] "clase_ternaria"                                  
[31] "mrentabilidad_rank"                              
[32] "mrentabilidad_annual_rank"                       
[33] "mcomisiones_rank"                                
[34] "mactivos_margen_rank"                            
[35] "mpasivos_margen_rank"                            
[36] "mcuenta_corriente_rank"                          
[37] "mcaja_ahorro_rank"                               
[38] "mcuentas_saldo_rank"                             
[39] "mtarjeta_visa_consumo_rank"                      
[40] "mtarjeta_master_consumo_rank"                    
[41] "mprestamos_personales_rank"                      
[42] "mpayroll_rank"                                   
[43] "mttarjeta_visa_debitos_automaticos_rank"         
[44] "mcomisiones_mantenimiento_rank"                  
[45] "mtransferencias_recibidas_rank"                  
[46] "Master_mfinanciacion_limite_rank"                
[47] "Master_msaldototal_rank"                         
[48] "Master_mlimitecompra_rank"                       
[49] "Master_mconsumototal_rank"                       
[50] "Master_mpagominimo_rank"                         
[51] "Visa_mfinanciacion_limite_rank"                  
[52] "Visa_msaldototal_rank"                           
[53] "Visa_mlimitecompra_rank"                         
[54] "Visa_mconsumototal_rank"                         
[55] "Visa_mpagominimo_rank"                           
[56] "kmes"                                            
[57] "ctrx_quarter_normalizado"                        
[58] "cantidad_total_transacciones"                    
[59] "cantidad_total_transacciones_quarter"            
[60] "cantidad_total_transacciones_quarter_normalizado"
[61] "vm_status01"                                     
[62] "vm_status02"                                     
[63] "vm_status03"                                     
[64] "vm_status04"                                     
[65] "vm_status05"                                     
[66] "vm_status06"                                     
[67] "mv_status07"                                     
[68] "vm_Fvencimiento"                                 
[69] "vm_fultimo_cierre"                               
[70] "vm_fechaalta"                                    
[71] "vm_cconsumos"

In [32]:
# No se implementa Feature Engineering a partir de Random Forest

In [33]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


In [34]:
ncol(dataset)
colnames(dataset)

[1] 343

[1] "numero_de_cliente"                                      
  [2] "foto_mes"                                               
  [3] "internet"                                               
  [4] "cliente_edad"                                           
  [5] "cliente_antiguedad"                                     
  [6] "cproductos"                                             
  [7] "cdescubierto_preacordado"                               
  [8] "ctarjeta_visa"                                          
  [9] "ctarjeta_visa_transacciones"                            
 [10] "ctarjeta_master"                                        
 [11] "ctarjeta_master_transacciones"                          
 [12] "cprestamos_personales"                                  
 [13] "cpayroll_trx"                                           
 [14] "ccomisiones_mantenimiento"                              
 [15] "ccomisiones_otras"                                      
 [16] "ccallcenter_transacciones"                              
 [17] "thomebanking"                                           
 [18] "chomebanking_transacciones"                             
 [19] "ctrx_quarter"                                           
 [20] "Master_status"                                          
 [21] "Master_Fvencimiento"                                    
 [22] "Master_fultimo_cierre"                                  
 [23] "Master_fechaalta"                                       
 [24] "Master_cconsumos"                                       
 [25] "Visa_status"                                            
 [26] "Visa_Fvencimiento"                                      
 [27] "Visa_fultimo_cierre"                                    
 [28] "Visa_fechaalta"                                         
 [29] "Visa_cconsumos"                                         
 [30] "clase_ternaria"                                         
 [31] "mrentabilidad_rank"                                     
 [32] "mrentabilidad_annual_rank"                              
 [33] "mcomisiones_rank"                                       
 [34] "mactivos_margen_rank"                                   
 [35] "mpasivos_margen_rank"                                   
 [36] "mcuenta_corriente_rank"                                 
 [37] "mcaja_ahorro_rank"                                      
 [38] "mcuentas_saldo_rank"                                    
 [39] "mtarjeta_visa_consumo_rank"                             
 [40] "mtarjeta_master_consumo_rank"                           
 [41] "mprestamos_personales_rank"                             
 [42] "mpayroll_rank"                                          
 [43] "mttarjeta_visa_debitos_automaticos_rank"                
 [44] "mcomisiones_mantenimiento_rank"                         
 [45] "mtransferencias_recibidas_rank"                         
 [46] "Master_mfinanciacion_limite_rank"                       
 [47] "Master_msaldototal_rank"                                
 [48] "Master_mlimitecompra_rank"                              
 [49] "Master_mconsumototal_rank"                              
 [50] "Master_mpagominimo_rank"                                
 [51] "Visa_mfinanciacion_limite_rank"                         
 [52] "Visa_msaldototal_rank"                                  
 [53] "Visa_mlimitecompra_rank"                                
 [54] "Visa_mconsumototal_rank"                                
 [55] "Visa_mpagominimo_rank"                                  
 [56] "kmes"                                                   
 [57] "ctrx_quarter_normalizado"                               
 [58] "cantidad_total_transacciones"                           
 [59] "cantidad_total_transacciones_quarter"                   
 [60] "cantidad_total_transacciones_quarter_normalizado"       
 [61] "vm_status01"                                            
 [62] "vm_status02"                                            
 [63] "vm_status03"               

In [35]:
VPOS_CORTE <- c()

fganancia_lgbm_meseta <- function(probs, datos) {
  vlabels <- get_field(datos, "label")
  vpesos <- get_field(datos, "weight")

  tbl <- as.data.table(list(
    "prob" = probs,
    "gan" = ifelse(vlabels == 1 & vpesos > 1, PARAM$CN$train$gan1, PARAM$CN$train$gan0)
  ))

  setorder(tbl, -prob)
  tbl[, posicion := .I]
  tbl[, gan_acum := cumsum(gan)]
  setorder(tbl, -gan_acum) # voy por la meseta

  gan <- mean(tbl[1:500, gan_acum]) # meseta de tamaño 500

  pos_meseta <- tbl[1:500, median(posicion)]
  VPOS_CORTE <<- c(VPOS_CORTE, pos_meseta)

  return(list(
    "name" = "ganancia",
    "value" = gan,
    "higher_better" = TRUE
  ))
}


In [36]:
# Elimina del dataset las variables que estan por debajo
#  de la capa geologica de canaritos
# se llama varias veces, luego de agregar muchas variables nuevas,
#  para ir reduciendo la cantidad de variables
# y así hacer lugar a nuevas variables importantes

GVEZ <- 1

campitos <- c( "numero_de_cliente", "foto_mes", "clase_ternaria" )

CanaritosAsesinos <- function(
  canaritos_ratio,
  canaritos_desvios,
  canaritos_semilla) {

  cat( "inicio CanaritosAsesinos()\n")
  gc(verbose= FALSE)
  dataset[, clase01 := 0L ]
  dataset[ clase_ternaria %in% PARAM$CN$train$clase01_valor1,
      clase01 := 1L ]

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  for (i in 1:(ncol(dataset) * canaritos_ratio)) {
    dataset[, paste0("canarito", i) := runif(nrow(dataset))]
  }

  campos_buenos <- setdiff(
    colnames(dataset),
    c( campitos, "clase01")
  )

  azar <- runif(nrow(dataset))

  dataset[, entrenamiento :=
    as.integer( foto_mes %in% PARAM$CN$train$training &
      (clase01 == 1 | azar < PARAM$CN$train$undersampling))]

  dtrain <- lgb.Dataset(
    data = data.matrix(dataset[entrenamiento == TRUE, campos_buenos, with = FALSE]),
    label = dataset[entrenamiento == TRUE, clase01],
    weight = dataset[
      entrenamiento == TRUE,
      ifelse(clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )

  dvalid <- lgb.Dataset(
    data = data.matrix(dataset[foto_mes %in% PARAM$CN$train$validation, campos_buenos, with = FALSE]),
    label = dataset[foto_mes %in% PARAM$CN$train$validation, clase01],
    weight = dataset[
      foto_mes %in% PARAM$CN$train$validation,
      ifelse( clase_ternaria %in% PARAM$CN$train$positivos, 1.0000001, 1.0)
    ],
    free_raw_data = FALSE
  )


  param <- list(
    objective = "binary",
    metric = "custom",
    first_metric_only = TRUE,
    boost_from_average = TRUE,
    feature_pre_filter = FALSE,
    verbosity = -100,
    seed = canaritos_semilla,
    max_depth = -1, # -1 significa no limitar,  por ahora lo dejo fijo
    min_gain_to_split = 0.0, # por ahora, lo dejo fijo
    lambda_l1 = 0.0, # por ahora, lo dejo fijo
    lambda_l2 = 0.0, # por ahora, lo dejo fijo
    max_bin = 31, # por ahora, lo dejo fijo
    num_iterations = 9999, # un numero grande, lo limita early_stopping_rounds
    force_row_wise = TRUE, # para que los alumnos no se atemoricen con  warning
    learning_rate = 0.065,
    feature_fraction = 1.0, # lo seteo en 1
    min_data_in_leaf = 260,
    num_leaves = 60,
    early_stopping_rounds = 200,
    num_threads = 1
  )

  set.seed(canaritos_semilla, kind = "L'Ecuyer-CMRG")
  modelo <- lgb.train(
    data = dtrain,
    valids = list(valid = dvalid),
    eval = fganancia_lgbm_meseta,
    param = param,
    verbose = -100
  )

  tb_importancia <- lgb.importance(model = modelo)
  tb_importancia[, pos := .I]

  fwrite(tb_importancia,
    file = paste0("impo_", GVEZ, ".txt"),
    sep = "\t"
  )

  GVEZ <<- GVEZ + 1

  umbral <- tb_importancia[
    Feature %like% "canarito",
    median(pos) + canaritos_desvios * sd(pos)
  ] # Atencion corto en la mediana mas desvios!!

  col_utiles <- tb_importancia[
    pos < umbral & !(Feature %like% "canarito"),
    Feature
  ]

  col_utiles <- unique(c(
    col_utiles,
    c(campitos, "mes")
  ))

  col_inutiles <- setdiff(colnames(dataset), col_utiles)

  dataset[, (col_inutiles) := NULL]

  cat( "fin CanaritosAsesinos()\n")

  return( tb_importancia )
}


In [37]:
# Estos DOS parametros son los que se deben modificar
PARAM$CN$ratio <- 0.2
PARAM$CN$desvios <- 2


# Parametros quasi fijos
# Parametros de un LightGBM que se genera para estimar la column importance
PARAM$CN$train$clase01_valor1 <- c( "BAJA+2", "BAJA+1")
PARAM$CN$train$positivos <- c( "BAJA+2")
PARAM$CN$train$training <- c( 202101, 202102, 202103)
PARAM$CN$train$validation <- c( 202105 )
PARAM$CN$train$undersampling <- 0.1
PARAM$CN$train$gan1 <- 0.975
PARAM$CN$train$gan0 <- -0.025

In [38]:
library(lightgbm)

In [39]:
# la llamada a Canaritos Asesinos
tb_importancia <- CanaritosAsesinos(
  canaritos_ratio = PARAM$CN$ratio,
  canaritos_desvios = PARAM$CN$desvios,
  canaritos_semilla = PARAM$semilla_primigenia
)


inicio CanaritosAsesinos()
fin CanaritosAsesinos()


In [40]:
# grabo la importancia, ver el archivo directamente en la carpeta

fwrite( tb_importancia,
  file="canaritos.txt",
  sep="\t"
)

In [41]:
# verifico
ncol(dataset)
colnames(dataset)

[1] 137

[1] "numero_de_cliente"                                      
  [2] "foto_mes"                                               
  [3] "cliente_edad"                                           
  [4] "cproductos"                                             
  [5] "cdescubierto_preacordado"                               
  [6] "ctarjeta_visa_transacciones"                            
  [7] "cprestamos_personales"                                  
  [8] "cpayroll_trx"                                           
  [9] "ccomisiones_mantenimiento"                              
 [10] "ccomisiones_otras"                                      
 [11] "ctrx_quarter"                                           
 [12] "Master_Fvencimiento"                                    
 [13] "Master_fultimo_cierre"                                  
 [14] "Master_fechaalta"                                       
 [15] "Visa_fultimo_cierre"                                    
 [16] "Visa_fechaalta"                                         
 [17] "clase_ternaria"                                         
 [18] "mrentabilidad_rank"                                     
 [19] "mrentabilidad_annual_rank"                              
 [20] "mactivos_margen_rank"                                   
 [21] "mpasivos_margen_rank"                                   
 [22] "mcuenta_corriente_rank"                                 
 [23] "mcaja_ahorro_rank"                                      
 [24] "mcuentas_saldo_rank"                                    
 [25] "mtarjeta_visa_consumo_rank"                             
 [26] "mtarjeta_master_consumo_rank"                           
 [27] "mprestamos_personales_rank"                             
 [28] "mpayroll_rank"                                          
 [29] "mcomisiones_mantenimiento_rank"                         
 [30] "Visa_mfinanciacion_limite_rank"                         
 [31] "Visa_msaldototal_rank"                                  
 [32] "Visa_mlimitecompra_rank"                                
 [33] "Visa_mpagominimo_rank"                                  
 [34] "ctrx_quarter_normalizado"                               
 [35] "cantidad_total_transacciones_quarter"                   
 [36] "vm_status01"                                            
 [37] "vm_status02"                                            
 [38] "vm_fultimo_cierre"                                      
 [39] "cliente_edad_lag1"                                      
 [40] "cliente_antiguedad_lag1"                                
 [41] "ctrx_quarter_lag1"                                      
 [42] "Master_Fvencimiento_lag1"                               
 [43] "Master_fechaalta_lag1"                                  
 [44] "mrentabilidad_annual_rank_lag1"                         
 [45] "mcomisiones_rank_lag1"                                  
 [46] "mactivos_margen_rank_lag1"                              
 [47] "mpasivos_margen_rank_lag1"                              
 [48] "mcaja_ahorro_rank_lag1"                                 
 [49] "mcuentas_saldo_rank_lag1"                               
 [50] "mprestamos_personales_rank_lag1"                        
 [51] "mttarjeta_visa_debitos_automaticos_rank_lag1"           
 [52] "mcomisiones_mantenimiento_rank_lag1"                    
 [53] "mtransferencias_recibidas_rank_lag1"                    
 [54] "Visa_mfinanciacion_limite_rank_lag1"                    
 [55] "Visa_mlimitecompra_rank_lag1"                           
 [56] "Visa_mconsumototal_rank_lag1"                           
 [57] "Visa_mpagominimo_rank_lag1"                             
 [58] "kmes_lag1"                                              
 [59] "cantidad_total_transacciones_quarter_lag1"              
 [60] "vm_Fvencimiento_lag1"                                   
 [61] "cliente_edad_lag2"                                      
 [62] "cliente_antiguedad_lag2"                                
 [63] "cpayroll_trx_lag2"         

In [42]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105
)


PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [43]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [44]:
# Agregamos el ID y el mes a la lista de columnas a ignorar
campos_buenos <- copy( setdiff(
    colnames(dataset), 
    c("clase_ternaria", "clase01", "azar", "numero_de_cliente", "foto_mes")
))

In [45]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [46]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 32938

In [47]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  lambda_l1 = 0.2, # nuevo
  lambda_l2 = 0.2,    # nuevo
  bagging_freq = 1, # nuevo
  bagging_fraction = 0.7, # nuevo
  learning_rate= 0.02,
  feature_fraction= 0.2,
  num_iterations= 2500,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 400,
  num_leaves= 64,
  min_data_in_leaf= 128,
  max_depth = 7
)


In [48]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  # hago espacio en la memoria
  niter <- modelo_train$best_iter
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

In [49]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
#ESTO FUE MODIFICADO VARIAS VECES.. 
tb_nueva <- CJ(num_leaves = c(70,89),
  min_data_in_leaf = c(400,500,600),
  bagging_fraction = c(0.7,0.6),
  learning_rate = c(0.1,0.2,0.04),
  num_iterations = c(2500,3000),
  feature_fraction= c(0.3,0.35),
  max_depth = c(5,6,7,8))

In [50]:
# registro a registro calculo la AUC
tb_nueva[,  c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Sun Jul 26 02:20:13 2026  70, 400, 0.6, 0.04, 2500, 0.3, 5 niter 402 AUC 0.930728269558349

Sun Jul 26 02:20:33 2026  70, 400, 0.6, 0.04, 2500, 0.3, 6 niter 165 AUC 0.930986200590644

Sun Jul 26 02:20:52 2026  70, 400, 0.6, 0.04, 2500, 0.3, 7 niter 104 AUC 0.930342430375143

Sun Jul 26 02:21:13 2026  70, 400, 0.6, 0.04, 2500, 0.3, 8 niter 130 AUC 0.93049442662784

Sun Jul 26 02:21:49 2026  70, 400, 0.6, 0.04, 2500, 0.35, 5 niter 447 AUC 0.93092071003133

Sun Jul 26 02:22:11 2026  70, 400, 0.6, 0.04, 2500, 0.35, 6 niter 150 AUC 0.929764415060265

Sun Jul 26 02:22:34 2026  70, 400, 0.6, 0.04, 2500, 0.35, 7 niter 152 AUC 0.932707293811135

Sun Jul 26 02:22:56 2026  70, 400, 0.6, 0.04, 2500, 0.35, 8 niter 87 AUC 0.931494297629281

Sun Jul 26 02:23:21 2026  70, 400, 0.6, 0.04, 3000, 0.3, 5 niter 402 AUC 0.930728269558349

Sun Jul 26 02:23:48 2026  70, 400, 0.6, 0.04, 3000, 0.3, 6 niter 165 AUC 0.930986200590644

Sun Jul 26 02:24:09 2026  70, 400, 0.6, 0.04, 3000, 0.3, 7 niter 104 AUC 0.9303

In [51]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,bagging_fraction,learning_rate,num_iterations,feature_fraction,max_depth,AUC
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
70,400,0.6,0.04,402,0.30,5,0.9307283
70,400,0.6,0.04,165,0.30,6,0.9309862
70,400,0.6,0.04,104,0.30,7,0.9303424
70,400,0.6,0.04,130,0.30,8,0.9304944
70,400,0.6,0.04,447,0.35,5,0.9309207
70,400,0.6,0.04,150,0.35,6,0.9297644
70,400,0.6,0.04,152,0.35,7,0.9327073
70,400,0.6,0.04,87,0.35,8,0.9314943
70,400,0.6,0.04,402,0.30,5,0.9307283


In [52]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 70

$min_data_in_leaf
[1] 500

$bagging_fraction
[1] 0.7

$learning_rate
[1] 0.1

$num_iterations
[1] 74

$feature_fraction
[1] 0.35

$max_depth
[1] 7

In [53]:
tb_nueva

num_leaves,min_data_in_leaf,bagging_fraction,learning_rate,num_iterations,feature_fraction,max_depth,AUC
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
70,500,0.7,0.10,74,0.35,7,0.9328542
70,500,0.7,0.10,74,0.35,7,0.9328542
70,600,0.6,0.04,134,0.35,8,0.9327817
70,600,0.6,0.04,134,0.35,8,0.9327817
70,400,0.6,0.04,152,0.35,7,0.9327073
70,400,0.6,0.04,152,0.35,7,0.9327073
89,600,0.7,0.04,137,0.30,7,0.9326956
89,600,0.7,0.04,137,0.30,7,0.9326956
89,500,0.7,0.10,55,0.30,7,0.9325960


## Final Training con semillero n=30 (unico cambio respecto del v7)

In [54]:
PARAM$trainingstrategy$final_train <- c(
  201901, 201902, 201903, 201904, 201905, 201906,
  201907, 201908, 201909, 201910, 201911, 201912,
  202001, 202002, 202003, 202004, 202005, 202006,
  202007, 202008, 202009, 202010, 202011, 202012,
  202101, 202102, 202103, 202104, 202105, 202106,
  202107
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train)

[1] 910853

In [55]:
# uno los parametros fijos y los mejores encontrados en el grid search
fijos <- copy(PARAM$lgbm$param_fijos)
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)
param_final

$objective
[1] "binary"

$metric
[1] "auc"

$first_metric_only
[1] TRUE

$boost_from_average
[1] TRUE

$feature_pre_filter
[1] FALSE

$verbosity
[1] -100

$force_row_wise
[1] TRUE

$seed
[1] 171719

$max_bin
[1] 31

$lambda_l1
[1] 0.2

$lambda_l2
[1] 0.2

$bagging_freq
[1] 1

$bagging_fraction
[1] 0.7

$learning_rate
[1] 0.02

$feature_fraction
[1] 0.2

$num_leaves
[1] 64

$min_data_in_leaf
[1] 128

$max_depth
[1] 7

$num_leaves
[1] 70

$min_data_in_leaf
[1] 500

$bagging_fraction
[1] 0.7

$learning_rate
[1] 0.1

$num_iterations
[1] 74

$feature_fraction
[1] 0.35

$max_depth
[1] 7

In [56]:
# genero el semillero a partir de la semilla primigenia
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
PARAM$FT$semillas <- sample(primos)[seq(PARAM$FT$semillerio)]

cat( PARAM$FT$semillas )

371069 314063 695867 963047 448627 169957 948989 872647 611449 242989 106411 641143 740141 740227 916913 667657 660811 683377 114553 246511 505823 777431 162641 119227 390581 853949 417019 659723 651877 288349

In [57]:
dir.create("modelos", showWarnings= FALSE)
primero <- TRUE

crear_modelo_final <- function( sem ) {

  # El nombre ahora incluye la cantidad de variables (campos_buenos)
  nombre_arch <- paste0( "./modelos/modelo_", length(campos_buenos), "_", sem, ".txt")
  if( !file.exists(nombre_arch) )   # checkpoint: si existe, no reentrena
  {
    param_semilla <- copy(param_final)
    param_semilla$seed <- sem

    set.seed(sem, kind = "L'Ecuyer-CMRG")
    final_model <- lgb.train(
      data= dfinal_train,
      param= param_semilla,
      verbose= -100
    )

    lgb.save(final_model, nombre_arch)

    if( primero )
    {
      primero <<- FALSE
      tb_importancia <- as.data.table(lgb.importance(final_model))
      fwrite( tb_importancia, file= "impo.txt", sep= "\t")
    }

    rm(final_model)
    gc(full = TRUE, verbose=FALSE)
  }
  cat("semilla", sem, "lista ", format(Sys.time(), "%X"), "\n")
}

gc(full = TRUE, verbose=FALSE)
for( sem in PARAM$FT$semillas)  crear_modelo_final(sem)

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2316480,123.8,4313695,230.4,4313695,230.4
Vcells,248868361,1898.8,423662621,3232.3,471538913,3597.6


semilla 371069 lista  05:41:01 
semilla 314063 lista  05:41:04 
semilla 695867 lista  05:41:07 
semilla 963047 lista  05:41:10 
semilla 448627 lista  05:41:12 
semilla 169957 lista  05:41:15 
semilla 948989 lista  05:41:18 
semilla 872647 lista  05:41:21 
semilla 611449 lista  05:41:24 
semilla 242989 lista  05:41:26 
semilla 106411 lista  05:41:29 
semilla 641143 lista  05:41:32 
semilla 740141 lista  05:41:42 
semilla 740227 lista  05:41:45 
semilla 916913 lista  05:41:48 
semilla 667657 lista  05:41:51 
semilla 660811 lista  05:41:53 
semilla 683377 lista  05:41:56 
semilla 114553 lista  05:41:59 
semilla 246511 lista  05:42:02 
semilla 505823 lista  05:42:05 
semilla 777431 lista  05:42:07 
semilla 162641 lista  05:42:10 
semilla 119227 lista  05:42:13 
semilla 390581 lista  05:42:16 
semilla 853949 lista  05:42:18 
semilla 417019 lista  05:42:21 
semilla 659723 lista  05:42:24 
semilla 651877 lista  05:42:27 
semilla 288349 lista  05:42:29 


## Prediccion ensemble + Submits a Kaggle

In [58]:
PARAM$trainingstrategy$future <- c(202109)
dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := 0]
datos_matrix <- data.matrix(dfuture[, campos_buenos, with= FALSE])

for( isem in seq(length(PARAM$FT$semillas)) )
{
  sem <- PARAM$FT$semillas[ isem ]
  # Debe coincidir exactamente con el formato anterior
  nombre_arch <- paste0( "./modelos/modelo_", length(campos_buenos), "_", sem, ".txt")
  final_model <- lgb.load(nombre_arch)
  prediccion <- predict(final_model, datos_matrix)
  tb_prediccion[, paste0("prob_", isem) := prediccion]
  tb_prediccion[, prob := prob + prediccion]
  rm(final_model); rm(prediccion)
  gc(full = TRUE, verbose=FALSE)
  cat("ensemble semilla", isem, "de", length(PARAM$FT$semillas), "\n")
}

rm( datos_matrix )
gc(full = TRUE, verbose=FALSE)
tb_prediccion[, prob := prob / length(PARAM$FT$semillas) ]

# guardo las probabilidades individuales y el promedio (util para ensembles futuros)
fwrite(tb_prediccion, file= "prediccion.txt", sep= "\t")

ensemble semilla 1 de 30 
ensemble semilla 2 de 30 
ensemble semilla 3 de 30 
ensemble semilla 4 de 30 
ensemble semilla 5 de 30 
ensemble semilla 6 de 30 
ensemble semilla 7 de 30 
ensemble semilla 8 de 30 
ensemble semilla 9 de 30 
ensemble semilla 10 de 30 
ensemble semilla 11 de 30 
ensemble semilla 12 de 30 
ensemble semilla 13 de 30 
ensemble semilla 14 de 30 
ensemble semilla 15 de 30 
ensemble semilla 16 de 30 
ensemble semilla 17 de 30 
ensemble semilla 18 de 30 
ensemble semilla 19 de 30 
ensemble semilla 20 de 30 
ensemble semilla 21 de 30 
ensemble semilla 22 de 30 
ensemble semilla 23 de 30 
ensemble semilla 24 de 30 
ensemble semilla 25 de 30 
ensemble semilla 26 de 30 
ensemble semilla 27 de 30 
ensemble semilla 28 de 30 
ensemble semilla 29 de 30 
ensemble semilla 30 de 30 


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,2328975,124.4,4313695,230.4,4313695,230.4
Vcells,131771154,1005.4,423662621,3232.3,471538913,3597.6


In [59]:
# genero archivos con los "envios" y los subo a Kaggle
PARAM$kaggle$competencia <- "data-mining-junior-2026-a"
PARAM$kaggle$cortes <- seq(1800, 2400, by = 50)

setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L]
  tb_prediccion[1:envios, Predicted := 1L]

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
    "  semilla=", PARAM$semilla_primigenia,
    "  exp=", PARAM$experimento,
    "  semillerio=", PARAM$FT$semillerio,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE)
  Sys.sleep(30)
  cat(salida, "\n")
}

Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 
Successfully submitted to Data Mining, Junior 2026 A 


In [60]:
if( !require("yaml")) install.packages("yaml")
require("yaml")
write_yaml( PARAM, file="PARAM.yml")

format(Sys.time(), "%a %b %d %X %Y")

Loading required package: yaml



[1] "Sun Jul 26 05:49:33 2026"